# 03 — Weaviate: industry-wise schema + Word/PDF upload with LangChain

> **Execution status at build time: NOT EXECUTED (`executed=false`).** The Weaviate image pull hit the Docker Hub anonymous rate limit.
> **Needs `podman login docker.io` then run**: `podman compose --profile weaviate up -d` (health: http://localhost:8080/v1/.well-known/ready) and execute this notebook.
> Code was verified statically against `weaviate-client` 4.24.1 (every code cell compiles; schema/filter/object payloads were built offline with the installed client).

Everything is in this notebook: schema, embedding, LangChain loading, ingestion gates, Weaviate store, tests.

| # | Section |
|---|---|
| 0 | Setup (uv `.venv` kernel) |
| 1 | Schema: base record + 8 industry models (Pydantic = single source of truth) |
| 2 | Embedding model (pinned, cached by content hash) |
| 3 | Load Word/PDF with **LangChain** + attach planned metadata from the manifest |
| 4 | Ingestion gates: source → purpose → classification → ACL → retention → sensitive scan → section chunking → validate |
| 5 | Weaviate store: one collection per industry (properties generated from the schema), self-provided vectors, HNSW cosine, **native multi-tenancy**, collection alias |
| 6 | Upload all industry documents (bad docs → quarantine) |
| 7 | Idempotency · 8 Staging → promote · 9 Safe search · 10 Isolation · 11 Update · 12 Delete · 13 Metrics |

Inputs: `data/docs/<industry>/*.pdf|*.docx` + `manifest.jsonl` (made by `scripts/make_docs.py`, all content fake).
System of record: per-notebook SQLite file `data/sor/weaviate.db` (redacted text only).
Design: [docs/PLAN.md](../docs/PLAN.md) · Setup: [SETUP.md](../SETUP.md)

## 0. Setup

In [ ]:
import os, re, json, math, time, hashlib, statistics, types, warnings
from datetime import date, datetime, timezone
from pathlib import Path
from typing import ClassVar, Literal, Union, get_args, get_origin
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # model already cached locally; delete this line on a new machine
DOCS, SAMPLES, RESULTS = ROOT / "data" / "docs", ROOT / "data" / "samples", ROOT / "data" / "results"
pd.set_option("display.max_colwidth", 90)

import sys; print("python:", sys.executable)

## 1. Schema — one record = one retrievable chunk

Base groups from the blueprint: **identity, provenance, content/vector, access/context, lifecycle**. Each industry adds its own typed fields and rules. Validation is the last gate: anything invalid never reaches the index.

In [ ]:
SCHEMA_VERSION = 1
TOKEN = r"^tok_[a-z0-9]+$"          # tokenized identifier, never a raw ID
Status = Literal["staged", "active", "superseded", "deleted"]
Classification = Literal["public", "internal", "confidential", "restricted"]


def make_chunk_id(tenant_id, source_id, source_version, chunk_no) -> str:
    """Deterministic -> idempotent upserts."""
    return hashlib.sha256(f"{tenant_id}|{source_id}|{source_version}|{chunk_no}".encode()).hexdigest()


def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


class BaseChunk(BaseModel):
    model_config = ConfigDict(extra="forbid")
    INDUSTRY: ClassVar[str] = ""
    # identity
    chunk_id: str
    tenant_id: str = Field(min_length=1)
    industry: str
    entity_type: str
    # provenance
    source_id: str
    source_version: str
    source_file: str
    chunk_no: int = Field(ge=0)
    section: str | None = None
    content_hash: str
    parent_id: str | None = None
    # content / vector
    text: str = Field(min_length=1)
    embedding: list[float]
    embedding_model: str
    language: str = "en"
    # access / context
    classification: Classification
    allowed_groups: list[str] = Field(min_length=1)        # fail closed: no ACL -> reject
    jurisdiction: str
    valid_from: date
    valid_to: date | None = None
    # lifecycle
    retention_until: date
    status: Status = "staged"
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

    @model_validator(mode="after")
    def _base_guardrails(self):
        if self.chunk_id != make_chunk_id(self.tenant_id, self.source_id, self.source_version, self.chunk_no):
            raise ValueError("chunk_id is not the stable hash of tenant|source|version|chunk_no")
        if self.content_hash != content_hash(self.text):
            raise ValueError("content_hash does not match text")
        if not self.embedding or not all(math.isfinite(x) for x in self.embedding):
            raise ValueError("embedding empty or non-finite")
        if self.valid_to and self.valid_to < self.valid_from:
            raise ValueError("valid_to before valid_from")
        if self.retention_until < date.today():
            raise ValueError("retention_until already passed")
        return self


BASE_FIELDS = frozenset(BaseChunk.model_fields)

In [ ]:
class HealthcareChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "healthcare"
    industry: Literal["healthcare"] = "healthcare"
    patient_ref_token: str = Field(pattern=TOKEN)
    record_type: str
    provider: str
    encounter_date: date
    consent_scope: Literal["treatment", "care_coordination", "research"]


class BankingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "banking"
    industry: Literal["banking"] = "banking"
    customer_ref_token: str | None = Field(default=None, pattern=TOKEN)
    product: str
    account_type: str
    document_type: str
    effective_date: date


class InsuranceChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "insurance"
    industry: Literal["insurance"] = "insurance"
    policy_ref: str = Field(pattern=TOKEN)
    claim_ref: str | None = Field(default=None, pattern=TOKEN)
    line_of_business: str
    event_date: date | None = None
    claim_status: Literal["open", "closed", "n/a"]


class LegalChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "legal"
    industry: Literal["legal"] = "legal"
    matter_id: str
    client_ref: str = Field(pattern=TOKEN)
    privilege_status: Literal["none", "privileged", "work_product"]
    document_type: str
    effective_date: date

    @model_validator(mode="after")
    def _privilege(self):
        if self.privilege_status != "none" and self.classification != "restricted":
            raise ValueError("privileged/work-product content must be classified 'restricted'")
        return self


class RetailChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "retail"
    industry: Literal["retail"] = "retail"
    product_id: str
    sku: str
    brand: str
    locale: str
    channel: Literal["online", "store", "omni"]
    price_valid_from: date | None = None
    price_valid_to: date | None = None
    inventory_region: str


class ManufacturingChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "manufacturing"
    industry: Literal["manufacturing"] = "manufacturing"
    asset_id: str
    site_id: str
    equipment_type: str
    safety_class: Literal["low", "medium", "high"]
    revision: str
    document_owner: str
    approval_status: Literal["approved"]          # draft/obsolete procedures are never indexed


class GovernmentChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "government"
    industry: Literal["government"] = "government"
    agency_id: str
    record_class: str
    release_status: Literal["public_release", "restricted_release"]
    case_ref_token: str | None = Field(default=None, pattern=TOKEN)

    @model_validator(mode="after")
    def _release(self):
        if self.classification == "public" and self.release_status != "public_release":
            raise ValueError("public classification requires release_status=public_release")
        return self


class EducationHrChunk(BaseChunk):
    INDUSTRY: ClassVar[str] = "education_hr"
    industry: Literal["education_hr"] = "education_hr"
    domain: Literal["education", "hr"]
    org_unit: str
    course_or_department: str
    term: str | None = None
    region: str
    subject_ref_token: str | None = Field(default=None, pattern=TOKEN)


INDUSTRIES: dict[str, type[BaseChunk]] = {m.INDUSTRY: m for m in (
    HealthcareChunk, BankingChunk, InsuranceChunk, LegalChunk,
    RetailChunk, ManufacturingChunk, GovernmentChunk, EducationHrChunk)}

print("base fields:", ", ".join(BaseChunk.model_fields))
pd.DataFrame([{"industry": i, "model": m.__name__,
               "industry fields": ", ".join(f for f in m.model_fields if f not in BASE_FIELDS)}
              for i, m in INDUSTRIES.items()])

## 2. Embedding model — pinned, normalized, cached by content hash

In [ ]:
from sentence_transformers import SentenceTransformer


class Embedder:
    QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge v1.5 query instruction

    def __init__(self, model_name="BAAI/bge-small-en-v1.5", tag="bgesmall"):
        self.model_name, self.tag = model_name, tag
        self.model = SentenceTransformer(model_name)
        self.dim = (getattr(self.model, "get_embedding_dimension", None) or self.model.get_sentence_embedding_dimension)()
        self._cache, self.cache_hits = {}, 0

    def embed(self, texts):
        keys = [content_hash(t) for t in texts]
        todo = [(k, t) for k, t in zip(keys, texts) if k not in self._cache]
        self.cache_hits += len(texts) - len(todo)
        if todo:
            for (k, _), v in zip(todo, self.model.encode([t for _, t in todo], normalize_embeddings=True)):
                self._cache[k] = v.astype(float).tolist()
        return [self._cache[k] for k in keys]

    def embed_query(self, q):
        return self.model.encode(self.QUERY_PREFIX + q, normalize_embeddings=True).astype(float).tolist()


embedder = Embedder()
print(embedder.model_name, "| dim", embedder.dim)

## 3. Load Word / PDF with LangChain + attach the planned metadata

- `PyPDFLoader` (one `Document` per page) and `Docx2txtLoader` (one `Document` per file).
- `manifest.jsonl` is the source system's metadata export: tenant, ACL, classification, jurisdiction, validity, retention, industry fields. It is merged into `Document.metadata`.
- A file **without** a manifest entry is rejected: no metadata = no ACL = fail closed.
- **Clean** step: drop repeated header/footer lines (banner, address/footer bars, signature line) so they never get embedded.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)          # langchain-community sunset notice
    from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

LOADERS = {".pdf": PyPDFLoader, ".docx": Docx2txtLoader}
CLASS_WORDS = {"PUBLIC", "INTERNAL", "CONFIDENTIAL", "RESTRICTED"}


@dataclass
class Rejection:
    industry: str
    source_id: str
    gate: str
    reason: str


def clean_text(text: str) -> str:
    """Blueprint 'CLEAN' step: remove page furniture, keep content."""
    keep = []
    for line in text.splitlines():
        s = line.strip()
        if not s or s in CLASS_WORDS:                       # empty / classification banner
            continue
        if "  |  " in s or s.startswith("Approved by:"):    # address bar, footer bar, signature line
            continue
        keep.append(re.sub(r"\s{2,}", " ", s))
    return "\n".join(keep)


def load_langchain_documents(industry: str):
    folder = DOCS / industry
    manifest = {m["file"]: m for m in (json.loads(l) for l in (folder / "manifest.jsonl").read_text("utf-8").splitlines() if l)}
    docs, rejections = [], []
    for path in sorted(p for p in folder.iterdir() if p.suffix.lower() in LOADERS):
        meta = manifest.get(path.name)
        if meta is None:
            rejections.append(Rejection(industry, path.stem, "metadata", f"{path.name} has no manifest entry")); continue
        loader = LOADERS[path.suffix.lower()]
        for page in loader(str(path)).load():
            page.metadata = {**page.metadata, **meta, "loader": loader.__name__}
            page.page_content = clean_text(page.page_content)
            docs.append(page)
    return docs, rejections


def to_source_docs(lc_docs: list[Document]) -> list[dict]:
    """Merge pages of the same file into one source doc (pipeline input)."""
    merged = {}
    for d in lc_docs:
        m = merged.setdefault(d.metadata["file"], {**{k: v for k, v in d.metadata.items()
                                                       if k not in ("page", "page_label", "total_pages")}, "text": ""})
        m["text"] = (m["text"] + "\n" + d.page_content).strip()
    return list(merged.values())


lc_docs, _ = load_langchain_documents("healthcare")
d0 = lc_docs[0]
print("page_content (cleaned):\n", d0.page_content[:600], "...\n")
{k: d0.metadata[k] for k in ["source", "loader", "file", "source_id", "tenant_id", "classification",
                             "allowed_groups", "jurisdiction", "valid_from", "retention_until", "metadata"]}

## 4. Ingestion gates (fail closed)

Order: **authorized source → approved purpose → classified → ACL present → retention valid → sensitive scan (reject secrets/SSN/cards, redact email/phone) → section-aware chunking (LangChain splitter) → embed → schema validation**. Each chunk text is prefixed with `title - section` so it stays meaningful on its own.

In [ ]:
APPROVED_SOURCES = {"ehr_export", "core_banking", "claims_system", "dms_legal", "pim", "plm", "records_mgmt", "lms_hris"}
APPROVED_PURPOSES = {"rag_search", "support_assistant"}
SECRETS = [re.compile(p) for p in (r"AKIA[0-9A-Z]{16}", r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
                                   r"(?i)\b(password|passwd|api[_-]?key|secret)\s*[:=]\s*\S+")]
SSN = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
CARD = re.compile(r"\b(?:\d[ -]?){13,19}\b")
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b")
PHONE = re.compile(r"\+?\d{1,2}[ .-]?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}\b")
SECTION = re.compile(r"^(\d+\. [A-Z][A-Za-z ]+|Revision History)$", re.M)

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40,
                                          separators=[". ", "; ", " ", ""], keep_separator="end")


def luhn(digits: str) -> bool:
    nums = [int(x) for x in digits][::-1]
    return sum(n if i % 2 == 0 else (n * 2 - 9 if n * 2 > 9 else n * 2) for i, n in enumerate(nums)) % 10 == 0


def sensitive_scan(text):
    """(reject_reason | None, redacted_text)"""
    if any(p.search(text) for p in SECRETS):
        return "credential/secret detected", text
    if SSN.search(text):
        return "SSN detected", text
    for m in CARD.finditer(text):
        digits = re.sub(r"\D", "", m.group())
        if 13 <= len(digits) <= 19 and luhn(digits):
            return "payment card number detected", text
    return None, PHONE.sub("[PHONE]", EMAIL.sub("[EMAIL]", text))


def split_sections(text, title):
    """[(section, body)] using the document's numbered headings; text before the first heading = control block."""
    marks = list(SECTION.finditer(text))
    parts = [("Document control", text[:marks[0].start()] if marks else text)]
    for i, m in enumerate(marks):
        parts.append((m.group(1), text[m.end(): marks[i + 1].start() if i + 1 < len(marks) else len(text)]))
    return [(s, " ".join(b.replace(title, "", 1).split() if s == "Document control" else b.split()))
            for s, b in parts if b.strip()]


@dataclass
class IngestResult:
    records: list = field(default_factory=list)
    rejections: list = field(default_factory=list)
    clean_docs: list = field(default_factory=list)


def run_pipeline(docs, industry, embedder) -> IngestResult:
    model, out = INDUSTRIES[industry], IngestResult()
    for doc in docs:
        sid = doc.get("source_id", "?")
        reject = lambda gate, why: out.rejections.append(Rejection(industry, sid, gate, why))
        if doc.get("source_system") not in APPROVED_SOURCES:
            reject("authorized_source", f"source_system '{doc.get('source_system')}' not approved"); continue
        if doc.get("purpose") not in APPROVED_PURPOSES:
            reject("approved_purpose", f"purpose '{doc.get('purpose')}' not approved"); continue
        if not doc.get("classification"):
            reject("classified", "missing classification"); continue
        if not doc.get("allowed_groups"):
            reject("acl", "no allowed_groups (ACL)"); continue
        if date.fromisoformat(doc["retention_until"]) < date.today():
            reject("retention", f"retention_until {doc['retention_until']} already passed"); continue
        why, text = sensitive_scan(doc["text"])
        if why:
            reject("sensitive_scan", why); continue

        pieces = [(sec, f"{doc['title']} - {sec}: {p}") for sec, body in split_sections(text, doc["title"])
                  for p in splitter.split_text(body)]
        vecs = embedder.embed([p for _, p in pieces])
        try:
            built = [model(
                chunk_id=make_chunk_id(doc["tenant_id"], sid, doc["source_version"], i),
                tenant_id=doc["tenant_id"], entity_type=doc["entity_type"], source_id=sid,
                source_version=doc["source_version"], source_file=doc["file"], chunk_no=i, section=sec,
                content_hash=content_hash(p), parent_id=sid, text=p, embedding=v,
                embedding_model=embedder.model_name, language=doc.get("language", "en"),
                classification=doc["classification"], allowed_groups=doc["allowed_groups"],
                jurisdiction=doc["jurisdiction"], valid_from=doc["valid_from"], valid_to=doc.get("valid_to"),
                retention_until=doc["retention_until"], **doc.get("metadata", {}))
                for i, ((sec, p), v) in enumerate(zip(pieces, vecs))]
        except ValidationError as e:
            err = e.errors()[0]
            reject("schema_validation", f"{'.'.join(map(str, err['loc'])) or 'record'}: {err['msg']}"); continue
        out.records += built
        out.clean_docs.append({**doc, "text": text})
    return out


demo = run_pipeline(to_source_docs(lc_docs), "healthcare", embedder)
pd.DataFrame([{"source_id": r.source_id, "chunk_no": r.chunk_no, "section": r.section, "text": r.text}
              for r in demo.records if r.source_id == "hc-001"])

## 5. Weaviate store — one collection per industry

Properties are **generated from the Pydantic model** (`str`/`Literal` → TEXT, `list[str]` → TEXT_ARRAY, `int` → INT, `date`/`datetime` → DATE as RFC3339 UTC so ranges compare natively).
Per industry: versioned collection `{Industry}_v1_bgesmall_384` (Weaviate requires a capitalized collection name), **self-provided vectors** (no vectorizer), HNSW cosine,
**native multi-tenancy** (one tenant = one shard per `tenant_id`, created on ingest), and an **alias** `{Industry}` (blue/green switch point, Weaviate ≥ 1.32 alias API; falls back to a documented dict if the server has no alias support).
Keyword properties (tenant, ACL groups, status, ids) use `FIELD` tokenization so equality / `contains_any` are exact — word tokenization would split `care_team_cardiology` into `care`, `team`, … and leak across groups.
Object UUID = `uuid5(chunk_id)` → idempotent upserts. `sources` + `quarantine` live in SQLite (system of record) — this notebook never touches the Postgres tables of 01.

In [ ]:
import sqlite3
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization, VectorDistances
from weaviate.classes.query import Filter, MetadataQuery
from weaviate.classes.tenants import Tenant
from weaviate.classes.data import DataObject
from weaviate.util import generate_uuid5
from weaviate.exceptions import WeaviateBaseError

SOR_PATH = ROOT / "data" / "sor" / "weaviate.db"
VECTOR = "default"                       # single named, self-provided vector
FULLTEXT = {"text", "section"}            # word-tokenized (BM25-able); every other TEXT prop is exact (FIELD)
SCALAR = {str: DataType.TEXT, int: DataType.INT, float: DataType.NUMBER, date: DataType.DATE, datetime: DataType.DATE}
ARRAY = {str: DataType.TEXT_ARRAY, int: DataType.INT_ARRAY, float: DataType.NUMBER_ARRAY, date: DataType.DATE_ARRAY}
RETURN = ["chunk_id", "tenant_id", "source_id", "source_version", "section", "text"]


def index_name(industry):
    return f"{industry.capitalize()}_v{SCHEMA_VERSION}_{embedder.tag}_{embedder.dim}"


def alias_name(industry):
    return industry.capitalize()


def weaviate_type(f):
    ann = f.annotation
    if get_origin(ann) in (Union, types.UnionType):
        ann = [a for a in get_args(ann) if a is not type(None)][0]
    if get_origin(ann) is Literal:
        return DataType.TEXT
    if get_origin(ann) is list:
        return ARRAY[get_args(ann)[0]]
    return SCALAR[ann]


def properties_for(model):
    props = []
    for name, f in model.model_fields.items():
        if name == "embedding":
            continue                                   # the vector itself, not a property
        dt = weaviate_type(f)
        is_text = dt in (DataType.TEXT, DataType.TEXT_ARRAY)
        props.append(Property(
            name=name, data_type=dt, index_filterable=True,
            index_range_filters=dt in (DataType.DATE, DataType.INT, DataType.NUMBER),
            tokenization=(Tokenization.WORD if name in FULLTEXT else Tokenization.FIELD) if is_text else None))
    return props


def as_utc(v):
    if isinstance(v, datetime):
        return v if v.tzinfo else v.replace(tzinfo=timezone.utc)
    if isinstance(v, date):
        return datetime(v.year, v.month, v.day, tzinfo=timezone.utc)
    return v


def to_properties(rec):
    """Record -> Weaviate properties: no vector, no nulls (absent = null), dates as UTC datetimes."""
    return {k: as_utc(v) for k, v in rec.model_dump().items() if k != "embedding" and v is not None}


def to_object(rec):
    return DataObject(uuid=generate_uuid5(rec.chunk_id), properties=to_properties(rec), vector={VECTOR: rec.embedding})


SOR_DDL = """
DROP TABLE IF EXISTS sources;
DROP TABLE IF EXISTS quarantine;
CREATE TABLE sources (
  source_id TEXT NOT NULL, source_version TEXT NOT NULL, tenant_id TEXT NOT NULL, industry TEXT NOT NULL,
  title TEXT, source_file TEXT, text TEXT NOT NULL, raw TEXT NOT NULL, retention_until TEXT NOT NULL,
  ingested_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP, PRIMARY KEY (source_id, source_version));
CREATE TABLE quarantine (
  id INTEGER PRIMARY KEY AUTOINCREMENT, industry TEXT, source_id TEXT, gate TEXT, reason TEXT,
  at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP, UNIQUE (industry, source_id, gate, reason));
"""


@dataclass
class Store:
    client: "weaviate.WeaviateClient"
    sor: sqlite3.Connection
    targets: dict = field(default_factory=dict)        # industry -> collection the alias points at
    alias_mode: str = "?"
    tenants: dict = field(default_factory=dict)        # industry -> set(tenant names)  (cache, filled on ingest)


def connect():
    host = os.getenv("WEAVIATE_HOST", "localhost")
    client = weaviate.connect_to_local(host=host, port=int(os.getenv("WEAVIATE_PORT", "8080")),
                                       grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "50051")))
    SOR_PATH.parent.mkdir(parents=True, exist_ok=True)
    return Store(client, sqlite3.connect(SOR_PATH, isolation_level=None))


def coll(conn, industry):
    return conn.client.collections.use(conn.targets[industry])


def create_indexes(conn):
    """Re-runnable: drops + recreates every versioned collection, its alias and the SQLite SoR tables."""
    conn.sor.executescript(SOR_DDL)
    for ind, model in INDUSTRIES.items():
        name, alias = index_name(ind), alias_name(ind)
        try:
            if conn.client.alias.get(alias_name=alias) is not None:
                conn.client.alias.delete(alias_name=alias)
        except WeaviateBaseError:
            pass                                           # server without alias support -> handled below
        if conn.client.collections.exists(name):
            conn.client.collections.delete(name)
        conn.client.collections.create(
            name,
            description=f"{ind} chunks, schema v{SCHEMA_VERSION}, {embedder.model_name}",
            properties=properties_for(model),
            vector_config=Configure.Vectors.self_provided(
                name=VECTOR, vector_index_config=Configure.VectorIndex.hnsw(distance_metric=VectorDistances.COSINE)),
            multi_tenancy_config=Configure.multi_tenancy(enabled=True, auto_tenant_creation=False),
            inverted_index_config=Configure.inverted_index(index_null_state=True))
        try:
            conn.client.alias.create(alias_name=alias, target_collection=name)
            conn.alias_mode = "server alias"
            conn.targets[ind] = conn.client.alias.get(alias_name=alias).collection   # resolve the pointer
        except WeaviateBaseError:
            conn.alias_mode = "client dict (server has no alias API)"
            conn.targets[ind] = name
        conn.tenants[ind] = set()
    return dict(conn.targets)


def ensure_tenants(conn, industry, names):
    col = coll(conn, industry)
    missing = set(names) - set(col.tenants.get())
    if missing:
        col.tenants.create([Tenant(name=t) for t in sorted(missing)])
    conn.tenants[industry] = set(col.tenants.get())


def save_sources(conn, industry, docs):
    conn.sor.executemany(
        "INSERT INTO sources (source_id, source_version, tenant_id, industry, title, source_file, text, raw, retention_until) "
        "VALUES (?,?,?,?,?,?,?,?,?) ON CONFLICT (source_id, source_version) DO UPDATE "
        "SET text = excluded.text, raw = excluded.raw, retention_until = excluded.retention_until",
        [(d["source_id"], d["source_version"], d["tenant_id"], industry, d.get("title"), d.get("file"), d["text"],
          json.dumps({k: v for k, v in d.items() if k != "text"}, default=str), str(d["retention_until"])) for d in docs])


def save_rejections(conn, rejections):
    conn.sor.executemany("INSERT OR IGNORE INTO quarantine (industry, source_id, gate, reason) VALUES (?,?,?,?)",
                         [(r.industry, r.source_id, r.gate, r.reason) for r in rejections])


def upsert(conn, industry, records):
    """Batch upsert per tenant shard; uuid5(chunk_id) makes a re-upload overwrite, never duplicate."""
    if not records:
        return 0
    by_tenant = {}
    for r in records:
        by_tenant.setdefault(r.tenant_id, []).append(r)
    ensure_tenants(conn, industry, by_tenant)
    col = coll(conn, industry)
    for t, recs in by_tenant.items():
        res = col.with_tenant(t).data.insert_many([to_object(r) for r in recs])
        if res.has_errors:
            raise RuntimeError(f"{industry}/{t}: {[e.message for e in list(res.errors.values())[:3]]}")
    return len(records)


def _set_status(conn, industry, flt, status):
    """Weaviate has no UPDATE ... WHERE: select matching uuids per shard, then partial-update (PATCH) each."""
    col, n = coll(conn, industry), 0
    for t in sorted(conn.tenants[industry]):
        shard = col.with_tenant(t)
        for o in shard.query.fetch_objects(filters=flt, limit=10_000, return_properties=["chunk_id"]).objects:
            shard.data.update(uuid=o.uuid, properties={"status": status}); n += 1
    return n


def promote(conn, industry, source_ids=None):
    flt = Filter.by_property("status").equal("staged")
    if source_ids:
        flt = flt & Filter.any_of([Filter.by_property("source_id").equal(s) for s in source_ids])
    return _set_status(conn, industry, flt, "active")


def supersede(conn, industry, source_id, keep_version):
    flt = Filter.all_of([Filter.by_property("source_id").equal(source_id),
                         Filter.by_property("source_version").not_equal(keep_version),
                         Filter.by_property("status").equal("active")])
    return _set_status(conn, industry, flt, "superseded")


def purge(conn, industry, source_id):
    col, n = coll(conn, industry), 0
    for t in sorted(conn.tenants[industry]):
        n += col.with_tenant(t).data.delete_many(where=Filter.by_property("source_id").equal(source_id)).successful
    conn.sor.execute("DELETE FROM sources WHERE source_id = ?", (source_id,))
    return n


def count(conn, industry, flt=None, tenant=None):
    col = coll(conn, industry)
    shards = [tenant] if tenant else sorted(conn.tenants[industry])
    return sum(col.with_tenant(t).aggregate.over_all(filters=flt, total_count=True).total_count for t in shards)


def search_filter(tenant_id, groups):
    today = as_utc(date.today())
    return Filter.all_of([
        Filter.by_property("tenant_id").equal(tenant_id),                 # defense in depth (the shard is already the tenant)
        Filter.by_property("status").equal("active"),
        Filter.by_property("allowed_groups").contains_any(list(groups)),  # ACL: any-match
        Filter.by_property("valid_from").less_or_equal(today),
        Filter.any_of([Filter.by_property("valid_to").is_none(True),
                       Filter.by_property("valid_to").greater_or_equal(today)])])


def safe_search(conn, industry, tenant_id, groups, query_vec, k=5):
    """Tenant = shard (with_tenant); ACL + status + validity are a pre-filter (allow-list) inside the same near_vector query."""
    if not tenant_id or not groups:
        raise PermissionError("safe_search needs tenant_id and at least one group")
    if tenant_id not in conn.tenants[industry]:
        return []                                                          # unknown tenant -> nothing (fail closed)
    res = coll(conn, industry).with_tenant(tenant_id).query.near_vector(
        near_vector=list(query_vec), target_vector=VECTOR, limit=k, filters=search_filter(tenant_id, groups),
        return_metadata=MetadataQuery(distance=True), return_properties=RETURN)
    return [{**{c: o.properties.get(c) for c in RETURN}, "score": 1 - o.metadata.distance} for o in res.objects]


conn = connect()
print("Weaviate", conn.client.get_meta()["version"], "| client", weaviate.__version__, "| ready:", conn.client.is_ready())
print(index_name("healthcare"), "properties:")
pd.DataFrame([{"property": p.name, "type": p.dataType.value,
               "tokenization": p.tokenization.value if p.tokenization else "-"} for p in properties_for(HealthcareChunk)])

In [ ]:
tables = create_indexes(conn)
print("alias mode:", conn.alias_mode)
pd.DataFrame([{"industry": i, "alias": alias_name(i), "collection": t,
               "multi-tenancy": conn.client.collections.use(t).config.get().multi_tenancy_config.enabled,
               "vectorizer": "self-provided", "properties": len(conn.client.collections.use(t).config.get().properties)}
              for i, t in tables.items()])

## 6. Upload all industry documents (Word/PDF → LangChain → gates → Weaviate)

In [ ]:
results, summary = {}, []
for ind in INDUSTRIES:
    lc, missing_meta = load_langchain_documents(ind)
    docs = to_source_docs(lc)
    res = run_pipeline(docs, ind, embedder)
    res.rejections += missing_meta
    save_sources(conn, ind, res.clean_docs)
    upsert(conn, ind, res.records)
    save_rejections(conn, res.rejections)
    results[ind] = res
    summary.append({"industry": ind, "files": len(docs),
                    "pdf": sum(d["file"].endswith(".pdf") for d in docs), "docx": sum(d["file"].endswith(".docx") for d in docs),
                    "accepted": len(res.clean_docs), "chunks": len(res.records), "quarantined": len(res.rejections),
                    "tenants (shards)": ", ".join(sorted(conn.tenants[ind]))})
pd.DataFrame(summary)

In [ ]:
# quarantine (fail closed): every industry's bad document stopped at a gate
q_df = pd.DataFrame(conn.sor.execute("select industry, source_id, gate, reason from quarantine order by industry").fetchall(),
                    columns=["industry", "source_id", "gate", "reason"])
EXPECTED_GATES = {"healthcare": "sensitive_scan", "banking": "sensitive_scan", "insurance": "authorized_source",
                  "legal": "schema_validation", "retail": "sensitive_scan", "manufacturing": "schema_validation",
                  "government": "retention", "education_hr": "acl"}
assert len(q_df) == len(INDUSTRIES) and dict(zip(q_df["industry"], q_df["gate"])) == EXPECTED_GATES
assert q_df["source_id"].str.endswith("-bad").all()
assert conn.sor.execute("select count(*) from sources where source_id like '%-bad'").fetchone()[0] == 0
q_df

In [ ]:
# stored objects carry the industry-specific properties (typed)
cols = ["source_id", "source_file", "chunk_no", "section", "tenant_id", "record_type", "patient_ref_token",
        "consent_scope", "encounter_date", "allowed_groups", "status"]
hc = coll(conn, "healthcare")
pd.DataFrame([{c: o.properties.get(c) for c in cols}
              for t in sorted(conn.tenants["healthcare"])
              for o in hc.with_tenant(t).query.fetch_objects(limit=10_000, return_properties=cols).objects]
             ).sort_values(["source_id", "chunk_no"]).reset_index(drop=True)

In [ ]:
# redaction: the email inside hc-001.pdf never reaches the index or the system of record
txt = conn.sor.execute("select text from sources where source_id = 'hc-001'").fetchone()[0]
assert "@" not in txt and "[EMAIL]" in txt
hc001 = [o.properties["text"] for t in sorted(conn.tenants["healthcare"]) for o in hc.with_tenant(t).query.fetch_objects(
    filters=Filter.by_property("source_id").equal("hc-001"), limit=10_000, return_properties=["text"]).objects]
assert hc001 and not any("@" in x for x in hc001)
print(re.search(r"[^.]*\[EMAIL\][^.]*\.?", txt).group().strip())

## 7. Idempotency — same upload again, same object counts

In [ ]:
before = {i: count(conn, i) for i in INDUSTRIES}
for ind, res in results.items():
    upsert(conn, ind, res.records)
after = {i: count(conn, i) for i in INDUSTRIES}
assert before == after and sum(after.values()) == sum(len(r.records) for r in results.values())
print("idempotent ✔", after, "| embedding cache hits:", embedder.cache_hits)

## 8. Staging → promote — staged chunks are never searchable

In [ ]:
golden = json.loads((SAMPLES / "golden_queries.json").read_text())
staged_hits = {g["query"]: len(safe_search(conn, g["industry"], g["tenant_id"], g["groups"], embedder.embed_query(g["query"])))
               for g in golden}
assert sum(staged_hits.values()) == 0
print("hits while staged:", sum(staged_hits.values()))
print("promoted:", {i: promote(conn, i) for i in INDUSTRIES})
status_df = pd.DataFrame([{"industry": i, "status": s, "chunks": n} for i in INDUSTRIES for s in get_args(Status)
                          if (n := count(conn, i, Filter.by_property("status").equal(s)))])
assert set(status_df["status"]) == {"active"}
status_df

## 9. Safe search — golden queries per industry

In [ ]:
rows, latencies = [], []
for g in golden:
    qv = embedder.embed_query(g["query"])
    t0 = time.perf_counter()
    hits = safe_search(conn, g["industry"], g["tenant_id"], g["groups"], qv, k=5)
    latencies.append((time.perf_counter() - t0) * 1000)
    found = [h["source_id"] for h in hits]
    rows.append({"industry": g["industry"], "tenant": g["tenant_id"], "query": g["query"], "expected": g["expected_source_id"],
                 "top1": found[0] if found else None, "top1 section": hits[0]["section"] if hits else None,
                 "score": round(hits[0]["score"], 3) if hits else None,
                 "hit@1": found[:1] == [g["expected_source_id"]], "hit@5": g["expected_source_id"] in found})
search_df = pd.DataFrame(rows)
assert search_df["hit@5"].all(), search_df[~search_df["hit@5"]]
print(f"hit@1 = {search_df['hit@1'].mean():.2f} | hit@5 = {search_df['hit@5'].mean():.2f} | p50 latency = {statistics.median(latencies):.1f} ms")
search_df

In [ ]:
pd.DataFrame(safe_search(conn, "manufacturing", "forgeworks", ["maintenance"], embedder.embed_query("how to lock out the hydraulic press")))

## 10. Isolation tests
**What enforces isolation here:** Weaviate **native multi-tenancy**. Each `tenant_id` is its own shard; every read/write on a multi-tenant collection must name a tenant (`.with_tenant()`), and the server rejects requests without one. A query in tenant A's shard physically cannot see tenant B's objects. The `tenant_id` property filter in `safe_search` is defense in depth only. ACL groups, status and validity are a **pre-filter**: Weaviate builds an allow-list from the inverted index and the HNSW search only returns allowed objects (never filtered in Python after top-k).
1. **Shard probe**: per shard, objects whose `tenant_id` is a different tenant = 0 (no tenant filter, only the shard); a query **without** a tenant raises.
2. **Cross-tenant**: other tenant asks the same golden question → never gets the owner's chunks.
3. **ACL**: user without the group gets nothing. 4. **Fail closed**: no tenant/groups → `PermissionError`.

In [ ]:
probe = []
for ind in INDUSTRIES:
    col = coll(conn, ind)
    try:
        col.query.fetch_objects(limit=1); no_tenant = "returned rows (NOT isolated)"
    except WeaviateBaseError as e:
        no_tenant = "raises ✔"
    row = {"industry": ind, "query without tenant": no_tenant}
    for t in sorted(conn.tenants[ind]):
        row[f"{t}: own"] = count(conn, ind, Filter.by_property("tenant_id").equal(t), tenant=t)
        row[f"{t}: foreign"] = count(conn, ind, Filter.by_property("tenant_id").not_equal(t), tenant=t)
        assert row[f"{t}: own"] == count(conn, ind, tenant=t) > 0 and row[f"{t}: foreign"] == 0
    probe.append(row)
probe_df = pd.DataFrame(probe).fillna("-")
assert (probe_df["query without tenant"] == "raises ✔").all()
probe_df

In [ ]:
iso = []
for g in golden:
    other = ({d["tenant_id"] for d in results[g["industry"]].clean_docs} - {g["tenant_id"]}).pop()
    qv = embedder.embed_query(g["query"])
    hits = safe_search(conn, g["industry"], other, g["groups"], qv)
    leaked = [h for h in hits if h["tenant_id"] != other or h["source_id"] == g["expected_source_id"]]
    no_acl = safe_search(conn, g["industry"], g["tenant_id"], ["group_nobody_has"], qv)
    iso.append({"industry": g["industry"], "owner": g["tenant_id"], "asked as": other,
                "hits for other tenant": len(hits), "leaked": len(leaked), "hits without ACL group": len(no_acl)})
iso_df = pd.DataFrame(iso)
assert iso_df["leaked"].eq(0).all() and iso_df["hits without ACL group"].eq(0).all()
for bad in [("", ["care_team_cardiology"]), ("acme_health", [])]:
    try:
        safe_search(conn, "healthcare", *bad, embedder.embed_query("x")); raise AssertionError("should raise")
    except PermissionError as e:
        print("fail closed ✔", e)
iso_df

## 11. Update — new version of `hc-002.docx` supersedes v1

In [ ]:
v1 = next(d for d in to_source_docs(load_langchain_documents("healthcare")[0]) if d["source_id"] == "hc-002")
v2 = {**v1, "source_version": "2",
      "text": v1["text"].replace("Metformin dose increased to 1000 mg twice daily", "Metformin dose increased to 1500 mg extended release once daily")}
res2 = run_pipeline([v2], "healthcare", embedder)
save_sources(conn, "healthcare", res2.clean_docs)
upsert(conn, "healthcare", res2.records)
promote(conn, "healthcare", ["hc-002"])
print("superseded chunks:", supersede(conn, "healthcare", "hc-002", keep_version="2"))
hits = safe_search(conn, "healthcare", "acme_health", ["care_team_endocrine"], embedder.embed_query("metformin dose"))
assert hits and any(h["source_id"] == "hc-002" for h in hits)
assert all(h["source_version"] == "2" for h in hits if h["source_id"] == "hc-002")
hc002 = Filter.by_property("source_id").equal("hc-002")
n_ver = lambda v, s: count(conn, "healthcare", hc002 & Filter.by_property("source_version").equal(v)
                           & Filter.by_property("status").equal(s))
assert n_ver("1", "active") == 0 and n_ver("1", "superseded") > 0 and n_ver("2", "active") > 0
display(pd.DataFrame([{"version": v, "status": s, "chunks": n} for v in ("1", "2") for s in get_args(Status)
                      if (n := count(conn, "healthcare", hc002 & Filter.by_property("source_version").equal(v)
                                     & Filter.by_property("status").equal(s)))]))
pd.DataFrame(hits)[["source_id", "source_version", "section", "score", "text"]]

## 12. Delete / consent withdrawal — purge `hc-004` and verify absence

In [ ]:
q = embedder.embed_query("ejection fraction systolic dysfunction")
before = [h["source_id"] for h in safe_search(conn, "healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
purged = purge(conn, "healthcare", "hc-004")
after = [h["source_id"] for h in safe_search(conn, "healthcare", "beacon_clinic", ["care_team_cardiology"], q)]
left = conn.sor.execute("select count(*) from sources where source_id = 'hc-004'").fetchone()[0]
in_index = count(conn, "healthcare", Filter.by_property("source_id").equal("hc-004"))
assert "hc-004" in before and "hc-004" not in after and purged > 0 and left == 0 and in_index == 0
print(f"purged {purged} chunks | before: {sorted(set(before))} | after: {sorted(set(after))} | "
      f"objects left: {in_index} | SoR rows left: {left} ✔")

## 13. Metrics (for the cross-DB comparison notebook)

In [ ]:
metrics = {"db": "weaviate", "industries": len(INDUSTRIES), "chunks_total": sum(count(conn, i) for i in INDUSTRIES),
           "quarantined": sum(len(r.rejections) for r in results.values()),
           "hit@1": float(search_df["hit@1"].mean()), "hit@5": float(search_df["hit@5"].mean()),
           "latency_p50_ms": round(statistics.median(latencies), 2),
           "isolation": "PASS (native multi-tenancy: one shard per tenant_id, tenant required on every op; "
                        "tenant_id + ACL contains_any + status + validity pre-filter in near_vector)",
           "executed": True}
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "weaviate.json").write_text(json.dumps(metrics, indent=2))
conn.client.close(); conn.sor.close()
metrics

## Next: embedding model change (blue/green)
Never re-embed into the existing collections. Create `{Industry}_v1_<newmodel>_<dim>` with a new `Embedder`, re-create the tenants, re-embed from `sources` (SQLite system of record), run the golden queries, then repoint the alias with `client.alias.update(alias_name="{Industry}", new_target_collection=<new collection>)` and delete the old collection.